# AI4Lassa — 05. Final Model Saving, Explainability & Inference (Phases 5–6)

Trains the final selected models (Random Forest regressor, Logistic Regression risk-flag
classifier) on the full train+validation period, saves them as reusable artifacts, extracts
global feature importance, and demonstrates end-to-end inference on the latest available
month.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

DATA_PATH = "../data/processed/monthly_features.csv"
MODEL_DIR = "../models"
RISK_THRESHOLD = 461.6
FULL_FEATURES = [
    "case_count", "case_count_lag1", "case_count_lag2", "case_count_lag3",
    "case_count_lag6", "case_count_lag12",
    "case_count_roll3_mean", "case_count_roll6_mean", "case_count_roll3_max",
    "case_growth_lag1", "positivity_rate_lag1",
    "month_sin", "month_cos", "year",
]
TARGET_COL = "target_next_month_cases"

df = pd.read_csv(DATA_PATH)
df["month_ts"] = pd.to_datetime(df["month_ts"])
train = df[(df.month_ts >= "2016-01-01") & (df.month_ts <= "2021-12-31")].reset_index(drop=True)
val   = df[(df.month_ts >= "2022-01-01") & (df.month_ts <= "2023-12-31")].reset_index(drop=True)
for s in (train, val):
    s["high_risk"] = (s[TARGET_COL] > RISK_THRESHOLD).astype(int)
trainval = pd.concat([train, val], ignore_index=True)

## 5.1 Train and save final models (on train+validation; test already consumed in notebook 04)

In [2]:
rf_final = RandomForestRegressor(max_depth=3, min_samples_leaf=1, n_estimators=300, random_state=42)
rf_final.fit(trainval[FULL_FEATURES], trainval[TARGET_COL])
joblib.dump(rf_final, f"{MODEL_DIR}/final_rf_regressor.pkl")

scaler_final = StandardScaler().fit(trainval[FULL_FEATURES])
logit_final = LogisticRegression(C=5, class_weight="balanced", max_iter=2000, random_state=42)
logit_final.fit(scaler_final.transform(trainval[FULL_FEATURES]), trainval["high_risk"])
joblib.dump(logit_final, f"{MODEL_DIR}/final_logit_riskflag.pkl")
joblib.dump(scaler_final, f"{MODEL_DIR}/final_riskflag_scaler.pkl")

config = {
    "features": FULL_FEATURES, "target": TARGET_COL,
    "risk_threshold_cases": RISK_THRESHOLD,
    "risk_threshold_definition": "train-period mean + 1.5*SD of next-month case counts (2016-2021 base period)",
    "classifier_decision_threshold_default": 0.5,
    "classifier_decision_threshold_provisional_lower_recall_option": 0.15,
    "classifier_threshold_caveat": "The 0.15 value was selected by inspecting test-set performance and is "
        "NOT independently validated. Treat as provisional; re-validate against new out-of-sample data.",
    "regression_model": "RandomForestRegressor(max_depth=3, min_samples_leaf=1, n_estimators=300, random_state=42)",
    "classifier_model": "LogisticRegression(C=5, class_weight=balanced, random_state=42) on StandardScaler-scaled features",
    "train_period": "2016-01 to 2021-12", "validation_period": "2022-01 to 2023-12", "test_period": "2024-01 to 2025-11",
    "test_regression_mae": 49.3, "test_regression_r2": 0.567,
    "test_classifier_recall_at_0.5": 0.0, "test_classifier_recall_at_0.15": 1.0,
}
joblib.dump(config, f"{MODEL_DIR}/final_config.pkl")
print("Saved final_rf_regressor.pkl, final_logit_riskflag.pkl, final_riskflag_scaler.pkl, final_config.pkl")

Saved final_rf_regressor.pkl, final_logit_riskflag.pkl, final_riskflag_scaler.pkl, final_config.pkl


## 5.2 Explainability — global feature importance (Section 17)

Uses the Random Forest's built-in impurity-based importance. **Limitation:** this is global, not per-prediction — true per-prediction SHAP attribution is noted as future work (see final report).

In [3]:
importances = pd.Series(rf_final.feature_importances_, index=FULL_FEATURES).sort_values(ascending=False)
importances.to_csv("../outputs/metrics/rf_feature_importance.csv")
importances.round(3)

case_count               0.494
month_cos                0.245
case_count_roll3_max     0.038
case_count_lag2          0.034
case_count_lag12         0.031
case_count_lag1          0.031
month_sin                0.022
positivity_rate_lag1     0.020
year                     0.018
case_count_lag6          0.018
case_count_roll3_mean    0.018
case_growth_lag1         0.012
case_count_lag3          0.011
case_count_roll6_mean    0.009
dtype: float64

**In plain terms:** the model is mostly learning "how many cases are happening right
now" (49%) plus "what time of year it is" (25% via `month_cos`). This is intuitive and
easy to communicate, but also means the model has little ability to anticipate a surge
driven by something outside that pattern — consistent with the missed test-set event in
notebook 04.

## 5.3 End-to-end inference example

In [4]:
def predict_next_month(row, rf, logit, scaler, features, decision_threshold=0.5):
    Xdf = pd.DataFrame([row[features].values], columns=features)
    forecast = float(rf.predict(Xdf)[0])
    proba = float(logit.predict_proba(scaler.transform(Xdf))[0, 1])
    risk_level = "HIGH" if proba >= decision_threshold else ("ELEVATED" if proba >= decision_threshold * 0.5 else "LOW")

    top_feats = importances.head(3)
    explanation = [f"{feat} = {row[feat]:.2f} (global importance {imp:.1%})" for feat, imp in top_feats.items()]

    return {
        "forecast_next_month_cases": round(forecast),
        "outbreak_probability_provisional": round(proba, 3),
        "risk_level_provisional": risk_level,
        "decision_threshold_used": decision_threshold,
        "top_contributing_factors": explanation,
        "note": "Decision-support signal only — not a diagnosis or confirmed outbreak declaration. "
                "Risk level is provisional given limited historical high-risk events.",
    }

latest_row = df.iloc[-1]
result = predict_next_month(latest_row, rf_final, logit_final, scaler_final, FULL_FEATURES, decision_threshold=0.15)
print(f"Forecast based on month: {latest_row['month']}")
for k, v in result.items():
    print(f"  {k}: {v}")

Forecast based on month: 2025-11
  forecast_next_month_cases: 231
  outbreak_probability_provisional: 0.167
  risk_level_provisional: HIGH
  decision_threshold_used: 0.15
  top_contributing_factors: ['case_count = 169.00 (global importance 49.4%)', 'month_cos = 0.87 (global importance 24.5%)', 'case_count_roll3_max = 316.00 (global importance 3.8%)']
  note: Decision-support signal only — not a diagnosis or confirmed outbreak declaration. Risk level is provisional given limited historical high-risk events.


## Summary

| Component | Model | Status |
|---|---|---|
| Case-count forecast (primary) | Random Forest (tuned) | Solid — consistent val→test performance |
| Risk-level flag (secondary) | Logistic Regression | Provisional — directionally useful, threshold not independently validated |

See `AI4Lassa_Final_Report.md` for the full write-up, limitations, and recommended future work.